# **I) Scraping**

In [ ]:
import pandas as pd
import requests
import json
import random
import html

headers = {'User-Agent':'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/56.0.2924.87 Safari/537.36',
'Accept-Encoding': 'gzip, deflate'}

cookies_jar = requests.cookies.RequestsCookieJar()
cookies_jar.set('name', 'jerry', domain='dev2qa.com', path='/cookies')
cookies_jar.set('password', 'jerry888', domain='dev2qa.com', path='/cookies')

url = "https://api.stocktwits.com/api/2/streams/symbol/MACRO.json"

prox = {}

def get_fed_twits(since = 0, max = 0):
    #since:	Returns results with an ID greater than (more recent than) the specified ID.
    #max:	Returns results with an ID less than (older than) or equal to the specified ID.
    data = {
            'since': '{}'.format(since),
            'max': '{}'.format(max),
            }

    page = requests.get(url, cookies=cookies_jar,headers= headers,proxies=prox, params=data).content

    contentJson = json.loads(page.decode())

    if contentJson['response']['status'] == 200:
        print("return 200")

    elif contentJson['response']['status'] == 404:
        print("404 not found")

    elif contentJson['response']['status'] == 429:
        print("429 rate limit exceeded")

    list_twits = contentJson['messages']

    return list_twits


def process_line(twit):
    line = {}

    line["id_"] = twit["id"]
    line["date"] = twit["created_at"]
    line["user_id"] = twit["user"]["id"]
    line["user_name"] = twit["user"]["username"]

    line["symbols"] = [item['symbol'] for item in twit['symbols']]

    try:
        if twit["entities"]["sentiment"]["basic"] == "Bullish":
            line["sent_"] = "Bullish"
        elif twit["entities"]["sentiment"]["basic"] == "Bearish":
            line["sent_"] = "Bearish"
        else:
            line["sent_"] = ""
    except TypeError:
        line["sent_"] = ""

    try:
        line["nb_likes"] = len(twit["likes"]["user_ids"])
    except KeyError:
        line["nb_likes"] = 0

    try:
        line["nb_reshares"] = len(twit["reshares"]["user_ids"])
    except KeyError:
        line["nb_reshares"] = 0

    mentions = []
    for mntn in twit["mentioned_users"]:
        mntn = mntn.replace('@','')
        try:
            mentions.append(mntn)
        except TypeError:
            pass
    line["mentions"] = mentions

    line["body"] = str(html.unescape(twit["body"]))
    return line


In [ ]:
df_final = pd.DataFrame()

list_twits = get_fed_twits(max = "593427153")

for twit in list_twits:
    df_final = pd.concat( [df_final, pd.DataFrame([process_line(twit)] )] )

for i in range(0, 500):
    print("---")
    max_id = list_twits[-1]["id"]
    print( max_id)

    list_twits = get_fed_twits(max =max_id)

    for twit in list_twits:
        df_final = pd.concat([df_final, pd.DataFrame([process_line(twit)])])

df_final.to_excel("data/output_MACRO_3.xlsx", index=False)